# LeetCode 3000 — Maximum Area of Longest Diagonal Rectangle bằng Python

Link đề: https://leetcode.com/problems/maximum-area-of-longest-diagonal-rectangle/ · Độ khó: **Easy** · Chủ đề: Array

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def areaOfMaxDiagonal(self, dimensions: List[List[int]]) -> int:
        ...
```

Ghi chú: đề và các ví dụ được viết theo trí nhớ về đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code, nhưng nên đối chiếu lại với đề gốc nếu muốn khớp từng chữ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong danh sách các hình chữ nhật, tìm hình có **đường chéo dài nhất**; nếu nhiều hình **cùng độ dài đường chéo** thì chọn hình có **diện tích lớn nhất**, và trả về **diện tích** đó.
2. Nhớ **định lý Pythagoras**: đường chéo `d` của hình chữ nhật `l x w` thỏa `d² = l² + w²`; và hiểu vì sao so sánh **bình phương** `l² + w²` cũng đủ để so sánh `d`, giúp **tránh số thực** và sai số.
3. Biết **so sánh bộ (tuple) theo từ điển**: `(d², area)` lớn nhất chính là "đường chéo lớn nhất, hòa thì diện tích lớn nhất", và biết dùng `max(..., key=...)`.
4. Đi từ cách **dùng `sqrt` (số thực)** → **so sánh bình phương số nguyên bằng `if/elif`** → **`max` trên tuple** → **hai lượt: tìm đường chéo lớn nhất rồi diện tích lớn nhất trong nhóm**.
5. Tránh các lỗi kinh điển: so sánh `l + w` hoặc chỉ diện tích thay vì đường chéo, xử lý **hòa** sai (cập nhật diện tích kể cả khi nhỏ hơn), so sánh số thực bằng `==`, và trả về đường chéo thay vì diện tích.

## 2. Đề bài & ví dụ

Bạn được cho mảng hai chiều `dimensions` với `n` hàng, hàng thứ `i` là `[length_i, width_i]`: **chiều dài** và **chiều rộng** của hình chữ nhật thứ `i`.

Hãy trả về **diện tích** của hình chữ nhật có **đường chéo dài nhất**. Nếu **có nhiều hình** cùng độ dài đường chéo dài nhất, hãy trả về diện tích của hình có **diện tích lớn nhất** trong số đó.

**Ví dụ 1**

- Input: `dimensions = [[9, 3], [8, 6]]`
- Output: `48`
- Giải thích: hình 0 có đường chéo `√(9² + 3²) = √90 ≈ 9.487`. Hình 1 có đường chéo `√(8² + 6²) = √100 = 10`. Hình 1 có đường chéo dài hơn nên đáp án là diện tích của nó `8 · 6 = 48`.

**Ví dụ 2**

- Input: `dimensions = [[3, 4], [4, 3]]`
- Output: `12`
- Giải thích: cả hai hình có đường chéo `5`. Diện tích cả hai đều `12`, nên đáp án `12`.

**Ràng buộc:** `1 <= dimensions.length <= 100`; `dimensions[i].length == 2`; `1 <= dimensions[i][0], dimensions[i][1] <= 100`.

In [1]:
from typing import List
from math import sqrt, hypot
import random

examples = [
    ([[9, 3], [8, 6]], 48),
    ([[3, 4], [4, 3]], 12),
    ([[1, 7], [5, 5]], 25),
    ([[5, 5], [1, 7]], 25),
]
for dims, expected in examples:
    print("dimensions =", dims, "-> kỳ vọng", expected)

dimensions = [[9, 3], [8, 6]] -> kỳ vọng 48
dimensions = [[3, 4], [4, 3]] -> kỳ vọng 12
dimensions = [[1, 7], [5, 5]] -> kỳ vọng 25
dimensions = [[5, 5], [1, 7]] -> kỳ vọng 25


## 3. Kiến thức nền

### 3.1. Đường chéo hình chữ nhật và định lý Pythagoras

Đường chéo của hình chữ nhật `l x w` chia nó thành hai tam giác vuông có hai cạnh góc vuông `l`, `w`. Theo **Pythagoras**: `d² = l² + w²`, nên `d = √(l² + w²)`. Ví dụ hình `3 x 4` có `d = 5`.

In [2]:
l, w = 3, 4
print("d^2 =", l * l + w * w)             # 25
print("d =", sqrt(l * l + w * w))          # 5.0
print(hypot(l, w))                         # 5.0: hàm có sẵn tính sqrt(l^2 + w^2)

d^2 = 25
d = 5.0
5.0


### 3.2. So sánh bình phương thay cho so sánh căn

Với số **không âm**, `a < b` ⇔ `a² < b²`. Vì độ dài luôn không âm, so sánh `d` hai hình chữ nhật tương đương so sánh `d² = l² + w²`. Lợi ích: `l² + w²` là **số nguyên**, so sánh **chính xác**, không dính sai số của số thực.

In [3]:
d1_sq = 9 * 9 + 3 * 3                      # 90
d2_sq = 8 * 8 + 6 * 6                      # 100
print(d1_sq < d2_sq)                        # True
print(sqrt(d1_sq) < sqrt(d2_sq))            # True (cùng kết luận)

True
True


### 3.3. Số thực có thể gây sai số

Trong Python `0.1 + 0.2 != 0.3` do số thực nhị phân không biểu diễn chính xác mọi số thập phân. Với `sqrt` của số nguyên ở bài này sai số rất nhỏ nên thường vẫn đúng, nhưng thói quen an toàn là **tránh `==` trên số thực**: nếu bài chỉ cần so sánh, hãy so sánh số nguyên.

In [4]:
print(0.1 + 0.2 == 0.3, 0.1 + 0.2)          # False 0.30000000000000004
print(sqrt(2) * sqrt(2) == 2, sqrt(2) ** 2) # False 2.0000000000000004
print(sqrt(25) == 5.0)                      # True (nhưng không phải lúc nào cũng may mắn)

False 0.30000000000000004
False 2.0000000000000004
True


### 3.4. Giải nén hàng `for l, w in dimensions`

Mỗi phần tử của `dimensions` là một danh sách hai số. Có thể **giải nén (unpack)** ngay trong `for`: `for l, w in dimensions`, thay vì `dimensions[i][0]`, `dimensions[i][1]`.

In [5]:
dimensions = [[9, 3], [8, 6]]
for l, w in dimensions:
    print(f"dài={l} rộng={w} diện tích={l * w} d^2={l * l + w * w}")

dài=9 rộng=3 diện tích=27 d^2=90
dài=8 rộng=6 diện tích=48 d^2=100


### 3.5. So sánh tuple theo thứ tự từ điển

Python so sánh hai tuple **từng phần tử từ trái sang phải**: gặp phần tử khác nhau đầu tiên thì quyết định kết quả; nếu bằng nhau mới xét phần tử kế. Vì vậy `(d², area)` lớn nhất = `d²` lớn nhất, **hòa thì** `area` lớn nhất, đúng yêu cầu của đề.

In [6]:
print((100, 48) > (90, 27))                 # True: d^2 lớn hơn
print((50, 25) > (50, 7))                   # True: d^2 hòa, area lớn hơn
print((50, 7) == (50, 7))                   # True
print(max([(90, 27), (100, 48)]))           # (100, 48)

True
True
True
(100, 48)


### 3.6. `max` với `key` và `sorted`

`max(iterable, key=hàm)` trả về **phần tử** có `hàm(phần_tử)` lớn nhất. Dùng `key=lambda p: (p[0]**2 + p[1]**2, p[0] * p[1])` ta chọn hình chữ nhật đúng yêu cầu. Nếu nhiều phần tử có cùng key lớn nhất, `max` trả về cái **đầu tiên** gặp.

In [7]:
dims = [[9, 3], [8, 6]]
best = max(dims, key=lambda p: (p[0] ** 2 + p[1] ** 2, p[0] * p[1]))
print(best, best[0] * best[1])              # [8, 6] 48

[8, 6] 48


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với mỗi hình tính **hai con số**: `d²` (để so đường chéo) và `area` (để phân định khi hòa). Rồi chọn hình có cặp `(d², area)` **lớn nhất**. Với `dimensions = [[9, 3], [8, 6]]`:

| hình | `l` | `w` | `d² = l² + w²` | `area = l · w` |
|---|---|---|---|---|
| 0 | 9 | 3 | `81 + 9 = 90` | 27 |
| 1 | 8 | 6 | `64 + 36 = 100` | 48 |

`100 > 90` nên hình 1 thắng; đáp án `48`. Với `[[1, 7], [5, 5]]`: cả hai có `d² = 50` nhưng diện tích `7` và `25`; hòa đường chéo nên chọn diện tích lớn hơn: `25`.

In [8]:
for dims in ([[9, 3], [8, 6]], [[1, 7], [5, 5]]):
    print("dimensions =", dims)
    for i, (l, w) in enumerate(dims):
        print(f"  hình {i}: d^2={l * l + w * w:>3}, area={l * w:>3}")

dimensions = [[9, 3], [8, 6]]
  hình 0: d^2= 90, area= 27
  hình 1: d^2=100, area= 48
dimensions = [[1, 7], [5, 5]]
  hình 0: d^2= 50, area=  7
  hình 1: d^2= 50, area= 25


## 5. Cách 1 — Brute Force / baseline: tính đường chéo bằng `sqrt` (số thực)

**Ý tưởng.** Với mỗi hình tính `d = sqrt(l² + w²)` (số thực) và diện tích. Giữ `best_d` và `best_area`: nếu `d > best_d` thì thay cả hai; nếu `d == best_d` thì lấy diện tích lớn hơn.

In [9]:
def area_of_max_diagonal_float(dimensions: List[List[int]]) -> int:
    best_d = -1.0
    best_area = 0
    for l, w in dimensions:
        d = sqrt(l * l + w * w)                # đường chéo: số thực
        area = l * w
        if d > best_d:
            best_d, best_area = d, area         # đường chéo dài hơn: thay hết
        elif d == best_d:
            best_area = max(best_area, area)    # hòa đường chéo: lấy diện tích lớn hơn
    return best_area


for dims, expected in examples:
    print(dims, "->", area_of_max_diagonal_float(dims), "(kỳ vọng", expected, ")")

[[9, 3], [8, 6]] -> 48 (kỳ vọng 48 )
[[3, 4], [4, 3]] -> 12 (kỳ vọng 12 )
[[1, 7], [5, 5]] -> 25 (kỳ vọng 25 )
[[5, 5], [1, 7]] -> 25 (kỳ vọng 25 )


In [10]:
# Xem từng bước so sánh với số thực
best_d, best_area = -1.0, 0
for l, w in [[9, 3], [8, 6]]:
    d, area = sqrt(l * l + w * w), l * w
    print(f"hình {l}x{w}: d={d:.6f} area={area} | best_d trước={best_d:.6f} best_area trước={best_area}")
    if d > best_d:
        best_d, best_area = d, area
    elif d == best_d:
        best_area = max(best_area, area)
print("kết quả:", best_area)

hình 9x3: d=9.486833 area=27 | best_d trước=-1.000000 best_area trước=0
hình 8x6: d=10.000000 area=48 | best_d trước=9.486833 best_area trước=27
kết quả: 48


**Độ phức tạp (Cách 1).** Một lượt qua `n` hình: **Time `O(n)`**, **Space `O(1)`**.

**Nút thắt.** Đúng nhưng **dựa vào so sánh số thực** (`==` giữa hai số thực). Với ràng buộc ở đây thường may mắn đúng, nhưng đây là thói quen rủi ro. Mà ta chỉ cần *so sánh*, không cần giá trị `d` thật, nên có thể bỏ căn.

## 6. Ý tưởng thuật toán / bước đột phá

Vì `sqrt` là hàm **tăng nghiêm ngặt** trên số không âm, so sánh `d` giữa hai hình tương đương so sánh `d² = l² + w²`:

> `d_i > d_j` ⇔ `l_i² + w_i² > l_j² + w_j²`, và `d_i == d_j` ⇔ `l_i² + w_i² == l_j² + w_j²`.

`l² + w²` là **số nguyên**, so sánh `>` và `==` đều **chính xác**. Ngoài ra, "đường chéo lớn nhất, hòa thì diện tích lớn nhất" chính là **cực đại theo từ điển** của cặp `(d², area)`, nên có thể dùng một lệnh `max`.

## 7. Các cách tiếp theo

### Cách 2 — So sánh bình phương số nguyên bằng `if / elif`

**Giải thích.** Giống Cách 1 nhưng thay `d` bằng `d_sq = l² + w²` (số nguyên). Mọi so sánh `>` và `==` đều chính xác.

In [11]:
def area_of_max_diagonal_int(dimensions: List[List[int]]) -> int:
    best_sq = -1                                # d^2 lớn nhất đến giờ
    best_area = 0
    for l, w in dimensions:
        d_sq = l * l + w * w
        area = l * w
        if d_sq > best_sq:
            best_sq, best_area = d_sq, area
        elif d_sq == best_sq and area > best_area:
            best_area = area
    return best_area


for dims, expected in examples:
    print(dims, "->", area_of_max_diagonal_int(dims), "(kỳ vọng", expected, ")")

[[9, 3], [8, 6]] -> 48 (kỳ vọng 48 )
[[3, 4], [4, 3]] -> 12 (kỳ vọng 12 )
[[1, 7], [5, 5]] -> 25 (kỳ vọng 25 )
[[5, 5], [1, 7]] -> 25 (kỳ vọng 25 )


**Độ phức tạp Cách 2:** **Time `O(n)`**, **Space `O(1)`**.

### Cách 3 — `max` trên tuple `(d², area)`

**Giải thích.** Mỗi hình thành một tuple `(l² + w², l · w)`. `max` của các tuple là tuple có `d²` lớn nhất, hòa thì `area` lớn nhất (so sánh từ điển, mục 3.5). Lấy `[1]` là diện tích. Ngắn nhất.

In [12]:
def area_of_max_diagonal_max(dimensions: List[List[int]]) -> int:
    return max((l * l + w * w, l * w) for l, w in dimensions)[1]


def area_of_max_diagonal_key(dimensions: List[List[int]]) -> int:
    best = max(dimensions, key=lambda p: (p[0] ** 2 + p[1] ** 2, p[0] * p[1]))
    return best[0] * best[1]


for dims, expected in examples:
    print(dims, "->", area_of_max_diagonal_max(dims), area_of_max_diagonal_key(dims), "(kỳ vọng", expected, ")")

[[9, 3], [8, 6]] -> 48 48 (kỳ vọng 48 )
[[3, 4], [4, 3]] -> 12 12 (kỳ vọng 12 )
[[1, 7], [5, 5]] -> 25 25 (kỳ vọng 25 )
[[5, 5], [1, 7]] -> 25 25 (kỳ vọng 25 )


**Độ phức tạp Cách 3:** **Time `O(n)`**, **Space `O(1)`** (generator không tạo list).

### Cách 4 — Hai lượt: tìm đường chéo lớn nhất, rồi diện tích lớn nhất trong nhóm

**Giải thích.** Lượt 1: `m = max(l² + w²)`. Lượt 2: trong các hình có `l² + w² == m`, lấy `max(l · w)`. Cách làm tách bạch hai tiêu chí; hữu ích khi tiêu chí phụ thay đổi hoặc phức tạp.

In [13]:
def area_of_max_diagonal_two_pass(dimensions: List[List[int]]) -> int:
    m = max(l * l + w * w for l, w in dimensions)                      # lượt 1
    return max(l * w for l, w in dimensions if l * l + w * w == m)     # lượt 2


for dims, expected in examples:
    print(dims, "->", area_of_max_diagonal_two_pass(dims), "(kỳ vọng", expected, ")")

# Quan sát hai lượt với [[1,7],[5,5],[3,4]]
dims = [[1, 7], [5, 5], [3, 4]]
m = max(l * l + w * w for l, w in dims)
print("lượt 1: d^2 lớn nhất =", m)
print("lượt 2: các hình có d^2 =", m, "->", [(l, w, l * w) for l, w in dims if l * l + w * w == m])

[[9, 3], [8, 6]] -> 48 (kỳ vọng 48 )
[[3, 4], [4, 3]] -> 12 (kỳ vọng 12 )
[[1, 7], [5, 5]] -> 25 (kỳ vọng 25 )
[[5, 5], [1, 7]] -> 25 (kỳ vọng 25 )
lượt 1: d^2 lớn nhất = 50
lượt 2: các hình có d^2 = 50 -> [(1, 7, 7), (5, 5, 25)]


**Độ phức tạp Cách 4:** **Time `O(n)`** (hai lượt), **Space `O(1)`**.

In [14]:
# Code nộp LeetCode cho Cách 2 (số nguyên, một lượt, rõ ràng)
from typing import List

class Solution:
    def areaOfMaxDiagonal(self, dimensions: List[List[int]]) -> int:
        best_sq = -1
        best_area = 0
        for l, w in dimensions:
            d_sq = l * l + w * w
            area = l * w
            if d_sq > best_sq:
                best_sq, best_area = d_sq, area
            elif d_sq == best_sq and area > best_area:
                best_area = area
        return best_area

## 8. TRACE từng bước

Ta theo dõi (a) Cách 2 với `[[1, 7], [5, 5], [3, 4], [7, 1]]` (có hòa) và (b) Cách 3 (tuple) để thấy phép so sánh từ điển.

In [15]:
def trace_int(dimensions):
    best_sq, best_area = -1, 0
    print("dimensions =", dimensions)
    for l, w in dimensions:
        d_sq, area = l * l + w * w, l * w
        if d_sq > best_sq:
            action = "d^2 lớn hơn -> thay cả hai"
            best_sq, best_area = d_sq, area
        elif d_sq == best_sq and area > best_area:
            action = "hòa d^2, area lớn hơn -> thay area"
            best_area = area
        elif d_sq == best_sq:
            action = "hòa d^2, area không lớn hơn -> giữ"
        else:
            action = "d^2 nhỏ hơn -> bỏ qua"
        print(f"{l}x{w}: d^2={d_sq:>3} area={area:>3} | {action:<36} | best_sq={best_sq:>3} best_area={best_area:>3}")
    print("=> kết quả:", best_area)
    return best_area


assert trace_int([[1, 7], [5, 5], [3, 4], [7, 1]]) == 25
assert trace_int([[9, 3], [8, 6]]) == 48

dimensions = [[1, 7], [5, 5], [3, 4], [7, 1]]
1x7: d^2= 50 area=  7 | d^2 lớn hơn -> thay cả hai           | best_sq= 50 best_area=  7
5x5: d^2= 50 area= 25 | hòa d^2, area lớn hơn -> thay area   | best_sq= 50 best_area= 25
3x4: d^2= 25 area= 12 | d^2 nhỏ hơn -> bỏ qua                | best_sq= 50 best_area= 25
7x1: d^2= 50 area=  7 | hòa d^2, area không lớn hơn -> giữ   | best_sq= 50 best_area= 25
=> kết quả: 25
dimensions = [[9, 3], [8, 6]]
9x3: d^2= 90 area= 27 | d^2 lớn hơn -> thay cả hai           | best_sq= 90 best_area= 27
8x6: d^2=100 area= 48 | d^2 lớn hơn -> thay cả hai           | best_sq=100 best_area= 48
=> kết quả: 48


In [16]:
def trace_tuple(dimensions):
    best = None
    for l, w in dimensions:
        cur = (l * l + w * w, l * w)
        if best is None or cur > best:
            print(f"{l}x{w}: tuple {cur} > {best}  -> cập nhật")
            best = cur
        else:
            print(f"{l}x{w}: tuple {cur} <= {best} -> giữ")
    print("=> diện tích:", best[1])
    return best[1]


assert trace_tuple([[1, 7], [5, 5], [3, 4], [7, 1]]) == 25

1x7: tuple (50, 7) > None  -> cập nhật
5x5: tuple (50, 25) > (50, 7)  -> cập nhật
3x4: tuple (25, 12) <= (50, 25) -> giữ
7x1: tuple (50, 7) <= (50, 25) -> giữ
=> diện tích: 25


Bảng tóm tắt (Cách 2, `[[1, 7], [5, 5], [3, 4], [7, 1]]`):

| hình | `d²` | `area` | hành động | `best_sq` | `best_area` |
|---|---|---|---|---|---|
| `1x7` | 50 | 7 | `d²` lớn hơn | 50 | 7 |
| `5x5` | 50 | 25 | hòa `d²`, area lớn hơn | 50 | 25 |
| `3x4` | 25 | 12 | `d²` nhỏ hơn | 50 | 25 |
| `7x1` | 50 | 7 | hòa `d²`, area không lớn hơn | 50 | 25 |

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (so sánh bình phương số nguyên). Đúng class `Solution`, đúng method `areaOfMaxDiagonal(self, dimensions: List[List[int]]) -> int`.

In [17]:
from typing import List

class Solution:
    def areaOfMaxDiagonal(self, dimensions: List[List[int]]) -> int:
        best_sq = -1                                   # bình phương đường chéo lớn nhất đến giờ
        best_area = 0                                  # diện tích tương ứng
        for l, w in dimensions:
            d_sq = l * l + w * w                       # d^2 = l^2 + w^2 (số nguyên, so sánh chính xác)
            area = l * w
            if d_sq > best_sq:                         # đường chéo dài hơn hẳn
                best_sq, best_area = d_sq, area
            elif d_sq == best_sq and area > best_area: # cùng đường chéo: lấy diện tích lớn hơn
                best_area = area
        return best_area


sol = Solution()
for dims, expected in examples:
    print(dims, "->", sol.areaOfMaxDiagonal(dims))

[[9, 3], [8, 6]] -> 48
[[3, 4], [4, 3]] -> 12
[[1, 7], [5, 5]] -> 25
[[5, 5], [1, 7]] -> 25


## 10. Vì sao thuật toán đúng

- **Bình phương bảo toàn thứ tự.** Với `a, b >= 0`: `a > b ⇔ a² > b²` và `a = b ⇔ a² = b²`. Vì đường chéo `d = √(l² + w²) >= 0`, so sánh `d` giữa các hình tương đương so sánh `d² = l² + w²`, và phép so sánh số nguyên này chính xác.
- **Bất biến.** Sau khi xử lý `k` hình đầu, `(best_sq, best_area)` là cặp **lớn nhất theo từ điển** của `(d², area)` trong `k` hình đó: nếu hình mới có `d²` lớn hơn thì nó thắng; nếu `d²` bằng nhau thì thắng khi `area` lớn hơn; ngược lại giữ nguyên.
- **Khớp đề.** Cặp lớn nhất theo từ điển cho "đường chéo lớn nhất, hòa thì diện tích lớn nhất", và ta trả về thành phần diện tích.
- **Khởi tạo `best_sq = -1`.** Vì `d² >= 2 > -1` nên hình đầu tiên luôn thay thế giá trị khởi tạo.
- **Hòa nhiều hình.** Nhánh `elif` lặp lại cho từng hình hòa, nên diện tích kết quả là **max** diện tích của mọi hình có đường chéo lớn nhất, không phụ thuộc thứ tự.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(dimensions)`.

- **Cách 1 (`sqrt`):** Time `O(n)`, Space `O(1)`.
- **Cách 2 (bình phương số nguyên):** Time `O(n)`, Space `O(1)`.
- **Cách 3 (`max` trên tuple):** Time `O(n)`, Space `O(1)`.
- **Cách 4 (hai lượt):** Time `O(n)`, Space `O(1)`.
- Phải xem mọi hình nên `O(n)` là tối ưu. Ở đây `n <= 100`, `l, w <= 100` nên `d² <= 20000`, rất nhỏ.

## 12. Bộ test

In [18]:
sol = Solution()
impls = {
    "float": area_of_max_diagonal_float,
    "int": area_of_max_diagonal_int,
    "max": area_of_max_diagonal_max,
    "key": area_of_max_diagonal_key,
    "two_pass": area_of_max_diagonal_two_pass,
    "submit": sol.areaOfMaxDiagonal,
}

for dims, expected in examples:
    for key, fn in impls.items():
        got = fn([r[:] for r in dims])
        assert got == expected, (key, dims, got, expected)
print("PASS ví dụ của đề (mọi cách)")

PASS ví dụ của đề (mọi cách)


### 12.1. Các trường hợp biên

In [19]:
def check_all(dims, expected, label):
    for key, fn in impls.items():
        got = fn([r[:] for r in dims])
        assert got == expected, (label, key, got, expected)
    print("PASS", label, "->", expected)


check_all([[1, 1]], 1, "một hình vuông nhỏ nhất")
check_all([[100, 100]], 10000, "một hình vuông lớn nhất")
check_all([[3, 4]], 12, "một hình, 3-4-5")
check_all([[3, 4], [4, 3]], 12, "hai hình hòa hoàn toàn")
check_all([[1, 7], [5, 5]], 25, "hòa đường chéo (50), diện tích 7 và 25: chọn 25")
check_all([[5, 5], [1, 7]], 25, "như trên nhưng đổi thứ tự: vẫn 25")
check_all([[1, 8], [4, 7]], 28, "hòa d^2 = 65: diện tích 8 và 28")
check_all([[1, 100], [10, 10]], 100, "hình dài-mỏng d^2 = 10001 vượt hình vuông d^2 = 200")
check_all([[100, 1], [1, 100], [99, 99]], 9801, "99x99: d^2 = 19602 lớn hơn 100x1: d^2 = 10001")
check_all([[2, 3], [3, 2], [1, 3]], 6, "hai hình bằng nhau lớn nhất, hình thứ ba nhỏ hơn")
check_all([[10, 10], [14, 1]], 100, "d^2 = 200 so với 197: hình 10x10 thắng dù hình 14x1 dài hơn")

PASS một hình vuông nhỏ nhất -> 1
PASS một hình vuông lớn nhất -> 10000
PASS một hình, 3-4-5 -> 12
PASS hai hình hòa hoàn toàn -> 12
PASS hòa đường chéo (50), diện tích 7 và 25: chọn 25 -> 25
PASS như trên nhưng đổi thứ tự: vẫn 25 -> 25
PASS hòa d^2 = 65: diện tích 8 và 28 -> 28
PASS hình dài-mỏng d^2 = 10001 vượt hình vuông d^2 = 200 -> 100
PASS 99x99: d^2 = 19602 lớn hơn 100x1: d^2 = 10001 -> 9801
PASS hai hình bằng nhau lớn nhất, hình thứ ba nhỏ hơn -> 6
PASS d^2 = 200 so với 197: hình 10x10 thắng dù hình 14x1 dài hơn -> 100


### 12.2. Tính chất và phản ví dụ

In [20]:
# Tính chất 1: đáp án bằng diện tích của một hình nào đó
random.seed(11)
for _ in range(500):
    dims = [[random.randint(1, 100), random.randint(1, 100)] for _ in range(random.randint(1, 100))]
    ans = sol.areaOfMaxDiagonal(dims)
    assert ans in {l * w for l, w in dims}
print("PASS đáp án là diện tích của một hình trong danh sách")

# Tính chất 2: đổi thứ tự các hình không đổi đáp án
for _ in range(300):
    dims = [[random.randint(1, 20), random.randint(1, 20)] for _ in range(random.randint(1, 30))]
    shuffled = [r[:] for r in dims]
    random.shuffle(shuffled)
    assert sol.areaOfMaxDiagonal(dims) == sol.areaOfMaxDiagonal(shuffled)
print("PASS thứ tự các hình không quan trọng")

# Tính chất 3: đổi chỗ dài/rộng của từng hình không đổi đáp án
for _ in range(300):
    dims = [[random.randint(1, 20), random.randint(1, 20)] for _ in range(random.randint(1, 30))]
    swapped = [[w, l] for l, w in dims]
    assert sol.areaOfMaxDiagonal(dims) == sol.areaOfMaxDiagonal(swapped)
print("PASS đổi chiều dài/rộng không đổi đáp án")

# Tính chất 4: hình thắng thật sự có đường chéo >= mọi hình khác (kiểm bằng định nghĩa)
for _ in range(300):
    dims = [[random.randint(1, 100), random.randint(1, 100)] for _ in range(random.randint(1, 50))]
    ans = sol.areaOfMaxDiagonal(dims)
    longest = max(l * l + w * w for l, w in dims)
    candidates = [l * w for l, w in dims if l * l + w * w == longest]
    assert ans == max(candidates)
print("PASS đáp án = max diện tích trong các hình có đường chéo lớn nhất")

# Tính chất 5: đầu vào không bị sửa
dims = [[9, 3], [8, 6]]
copy = [r[:] for r in dims]
for key, fn in impls.items():
    fn(dims)
assert dims == copy
print("PASS đầu vào không bị sửa")

# Phản ví dụ 1: so sánh l + w (chu vi nửa) thay vì đường chéo
def sai_tong(dimensions):
    best = max(dimensions, key=lambda p: (p[0] + p[1], p[0] * p[1]))
    return best[0] * best[1]

assert sai_tong([[1, 9], [5, 5]]) == 25 and sol.areaOfMaxDiagonal([[1, 9], [5, 5]]) == 9
print("PASS phản ví dụ: so sánh l + w cho", sai_tong([[1, 9], [5, 5]]), "thay vì 9 ở [[1, 9], [5, 5]]")

# Phản ví dụ 2: hòa nhưng cập nhật diện tích bằng >= trên d^2 mà không so area
def sai_hoa(dimensions):
    best_sq, best_area = -1, 0
    for l, w in dimensions:
        if l * l + w * w >= best_sq:
            best_sq, best_area = l * l + w * w, l * w
    return best_area

assert sai_hoa([[5, 5], [1, 7]]) == 7 != 25
print("PASS phản ví dụ: >= trên d^2 ghi đè diện tích:", sai_hoa([[5, 5], [1, 7]]), "thay vì 25")

PASS đáp án là diện tích của một hình trong danh sách
PASS thứ tự các hình không quan trọng
PASS đổi chiều dài/rộng không đổi đáp án
PASS đáp án = max diện tích trong các hình có đường chéo lớn nhất
PASS đầu vào không bị sửa
PASS phản ví dụ: so sánh l + w cho 25 thay vì 9 ở [[1, 9], [5, 5]]
PASS phản ví dụ: >= trên d^2 ghi đè diện tích: 7 thay vì 25


### 12.3. Đối chiếu ngẫu nhiên (có seed)

In [21]:
random.seed(3000)
for trial in range(3000):
    n = random.randint(1, 100)
    hi = random.choice([3, 10, 30, 100])                  # miền nhỏ -> nhiều hòa đường chéo
    dims = [[random.randint(1, hi), random.randint(1, hi)] for _ in range(n)]
    ref = area_of_max_diagonal_two_pass([r[:] for r in dims])
    for key, fn in impls.items():
        assert fn([r[:] for r in dims]) == ref, (key, dims)
print("PASS 3000 bộ ngẫu nhiên (miền nhỏ gây nhiều hòa) - mọi cách khớp nhau")

# Toàn bộ miền ràng buộc: sqrt có làm hai giá trị d^2 khác nhau trùng nhau không?
sq_vals = sorted({l * l + w * w for l in range(1, 101) for w in range(1, 101)})
floats = [sqrt(v) for v in sq_vals]
assert all(floats[i] < floats[i + 1] for i in range(len(floats) - 1))
print(f"PASS trong miền 1..100: {len(sq_vals)} giá trị d^2 khác nhau cho {len(set(floats))} giá trị sqrt khác nhau (sqrt không làm trùng)")

PASS 3000 bộ ngẫu nhiên (miền nhỏ gây nhiều hòa) - mọi cách khớp nhau
PASS trong miền 1..100: 3678 giá trị d^2 khác nhau cho 3678 giá trị sqrt khác nhau (sqrt không làm trùng)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: So sánh `l + w` thay vì đường chéo

In [22]:
def sai_cong(dimensions):
    best = max(dimensions, key=lambda p: (p[0] + p[1], p[0] * p[1]))
    return best[0] * best[1]


dims = [[1, 9], [5, 5]]                                  # cùng l + w = 10; đường chéo: sqrt(82) và sqrt(50)
print("sai  :", sai_cong(dims), "| đúng:", Solution().areaOfMaxDiagonal(dims))

sai  : 25 | đúng: 9


**Vì sao sai:** `l + w` **không** đo đường chéo, đường chéo là `√(l² + w²)`. Hai hình có cùng `l + w = 10` thì tiêu chí hòa chuyển sang diện tích (`9` và `25`) và chọn `25`; nhưng hình `1 x 9` có đường chéo `√82` **dài hơn** hình `5 x 5` (`√50`), nên đáp án đúng là `9`. Phải dùng `l² + w²`.

### Lỗi 2: Chỉ chọn theo diện tích lớn nhất

In [23]:
def sai_dien_tich(dimensions):
    return max(l * w for l, w in dimensions)


print("sai  :", sai_dien_tich([[1, 100], [20, 20]]), "| đúng:", Solution().areaOfMaxDiagonal([[1, 100], [20, 20]]))

sai  : 400 | đúng: 100


**Vì sao sai:** tiêu chí **chính** là đường chéo; diện tích chỉ để phân định khi **hòa**. Hình `1 x 100` có `d² = 10001` lớn hơn hẳn hình `20 x 20` (`d² = 800`) dù diện tích chỉ `100` so với `400`. Đáp án đúng là `100`.

### Lỗi 3: Xử lý hòa sai (ghi đè diện tích bằng `>=`)

In [24]:
def sai_hoa_2(dimensions):
    best_sq, best_area = -1, 0
    for l, w in dimensions:
        if l * l + w * w >= best_sq:                     # hòa cũng ghi đè, không so diện tích
            best_sq, best_area = l * l + w * w, l * w
    return best_area


print("sai  :", sai_hoa_2([[5, 5], [1, 7]]), "| đúng:", Solution().areaOfMaxDiagonal([[5, 5], [1, 7]]))

sai  : 7 | đúng: 25


**Vì sao sai:** khi `d²` bằng nhau, đề yêu cầu giữ **diện tích lớn hơn**. Ghi đè bằng hình xuất hiện sau làm kết quả phụ thuộc thứ tự: ở `[[5, 5], [1, 7]]` hình `1 x 7` (diện tích `7`) ghi đè `5 x 5` (diện tích `25`). Cần nhánh riêng `elif d_sq == best_sq and area > best_area`.

### Lỗi 4: Trả về đường chéo (hoặc `d²`) thay vì diện tích

In [25]:
def sai_tra_duong_cheo(dimensions):
    best_sq = max(l * l + w * w for l, w in dimensions)
    return sqrt(best_sq)


print("sai  :", sai_tra_duong_cheo([[9, 3], [8, 6]]), "| đúng:", Solution().areaOfMaxDiagonal([[9, 3], [8, 6]]))

sai  : 10.0 | đúng: 48


**Vì sao sai:** đề hỏi **diện tích** (`48`), không phải đường chéo (`10.0`). Luôn đọc xem hàm cần trả về đại lượng nào, và kiểu của nó: ở đây là số nguyên `int`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "so sánh khoảng cách / độ dài đường chéo" → **so sánh bình phương** (số nguyên, chính xác, không cần `sqrt`).
- Công thức nhớ: **`d² = l² + w²`**, chọn hình có `(d², area)` **lớn nhất theo từ điển**.
- "Lớn nhất, hòa thì ..." → so sánh **tuple** `(tiêu_chí_chính, tiêu_chí_phụ)`; `max` làm hết việc.
- Tránh `==` giữa các số thực. Nếu bắt buộc dùng thực, so sánh với dung sai: `abs(a - b) < 1e-9`.
- `max(..., key=...)` trả về **phần tử đầu tiên** đạt cực đại; nếu đề phụ thuộc thứ tự khi hòa, hãy kiểm tra.
- Khởi tạo `best` bằng giá trị nhỏ hơn mọi khả năng (`-1` cho đại lượng `>= 1`).

## 15. Biến thể của bài

1. Trả về **chỉ số** của hình có đường chéo dài nhất (hòa thì diện tích lớn nhất, hòa nữa thì chỉ số nhỏ).
2. Trả về **độ dài** đường chéo dài nhất (số thực).
3. Trả về **top `k`** hình theo `(d², area)` giảm dần.
4. Hình **hộp chữ nhật** `l x w x h`: đường chéo không gian `√(l² + w² + h²)`.
5. Đường chéo lớn nhất mà diện tích **không vượt** một ngưỡng `T`.

Thử biến thể 1, 2 và 4:

In [26]:
def index_of_max_diagonal(dimensions):
    best_i, best = -1, None
    for i, (l, w) in enumerate(dimensions):
        cur = (l * l + w * w, l * w)
        if best is None or cur > best:                 # > nghiêm ngặt: hòa giữ chỉ số nhỏ hơn
            best, best_i = cur, i
    return best_i


def longest_diagonal_length(dimensions):
    return sqrt(max(l * l + w * w for l, w in dimensions))


def box_max_diagonal_volume(boxes):
    return max((l * l + w * w + h * h, l * w * h) for l, w, h in boxes)[1]


print("1.", index_of_max_diagonal([[5, 5], [1, 7], [7, 1]]))             # 0 (5x5 area 25 thắng)
print("2.", longest_diagonal_length([[9, 3], [8, 6]]))                     # 10.0
print("4.", box_max_diagonal_volume([[1, 2, 2], [2, 2, 1], [1, 1, 3]]))    # d^2 = 9, 9, 11 -> 3
assert index_of_max_diagonal([[5, 5], [1, 7], [7, 1]]) == 0
assert index_of_max_diagonal([[3, 4], [4, 3]]) == 0
assert longest_diagonal_length([[9, 3], [8, 6]]) == 10.0
assert box_max_diagonal_volume([[1, 2, 2], [2, 2, 1], [1, 1, 3]]) == 3

1. 0
2. 10.0
4. 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `sqrt` số thực | `d = sqrt(l² + w²)` | `O(n)` | `O(1)` | học định nghĩa; cẩn thận so sánh `==` |
| 2. Bình phương số nguyên, `if/elif` | `d² = l² + w²` | `O(n)` | `O(1)` | **cách nộp bài**, chính xác, rõ cơ chế hòa |
| 3. `max` trên tuple `(d², area)` | so sánh từ điển | `O(n)` | `O(1)` | ngắn gọn nhất |
| 4. Hai lượt | max `d²`, rồi max `area` trong nhóm | `O(n)` | `O(1)` | tách tiêu chí, dễ mở rộng |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `3 * 3 + 4 * 4`, `(10, 5) > (10, 3)`, `max([(2, 9), (3, 1)])`, `Solution().areaOfMaxDiagonal([[2, 2]])`, `Solution().areaOfMaxDiagonal([[6, 8], [10, 1]])`.

**Bài 2 (tính tay).** Với `dimensions = [[4, 7], [1, 8], [6, 6], [5, 5]]`: lập bảng `d²` và `area` cho từng hình, rồi cho biết đáp án.

**Bài 3 (điền TODO).** Hoàn thành `area_todo(dimensions)` theo Cách 2.

In [27]:
def area_todo(dimensions):
    best_sq = -1
    best_area = 0
    for l, w in dimensions:
        d_sq = l * l + w * w
        area = l * w
        # TODO 1: nếu d_sq > best_sq thì cập nhật cả best_sq và best_area
        # TODO 2: ngược lại nếu d_sq == best_sq và area > best_area thì cập nhật best_area
        pass
    return best_area

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (các TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `area_two_pass2(dimensions)` theo **hai lượt** (Cách 4) mà không nhìn code mẫu. Kiểm tra với bốn ví dụ trong `examples`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `top_k_rectangles(dimensions, k)` trả về `k` hình (dưới dạng `[l, w]`) tốt nhất theo `(d², area)` giảm dần. (b) Viết `count_ties(dimensions)` trả về **số hình** có đường chéo dài nhất. (c) Viết `max_diagonal_with_area_limit(dimensions, T)` trả về diện tích của hình có đường chéo dài nhất **trong các hình có diện tích `<= T`** (hòa thì diện tích lớn hơn); trả `0` nếu không hình nào thỏa.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `25`, `True`, `(3, 1)`, `4`, `10`. (`[[6, 8], [10, 1]]`: `d² = 100` và `101`, nên hình `10 x 1` thắng với diện tích `10`.) Bài 3: hai nhánh `if` / `elif` như trong code nộp.
- **Mức 2 (vừa):** Bài 2: `d²`: `4x7 = 65` (area 28); `1x8 = 65` (area 8); `6x6 = 72` (area 36); `5x5 = 50` (area 25). `d²` lớn nhất là `72` nên đáp án `36`. Bài 4: `m = max(l*l + w*w ...)` rồi `max(l*w for ... if l*l + w*w == m)`.
- **Mức 3 (sâu):** Bài 5(a): `sorted(dimensions, key=lambda p: (p[0]**2 + p[1]**2, p[0]*p[1]), reverse=True)[:k]`. 5(b): `m = max(...)` rồi đếm các hình có `d² == m`. 5(c): lọc trước `[(l, w) for l, w in dimensions if l * w <= T]`, rồi dùng `max` trên tuple; nhớ xử lý danh sách rỗng.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [28]:
# Bài 1
assert 3 * 3 + 4 * 4 == 25 and (10, 5) > (10, 3) and max([(2, 9), (3, 1)]) == (3, 1)
assert Solution().areaOfMaxDiagonal([[2, 2]]) == 4
assert Solution().areaOfMaxDiagonal([[6, 8], [10, 1]]) == 10

# Bài 2
assert [(l * l + w * w, l * w) for l, w in [[4, 7], [1, 8], [6, 6], [5, 5]]] == [(65, 28), (65, 8), (72, 36), (50, 25)]
assert Solution().areaOfMaxDiagonal([[4, 7], [1, 8], [6, 6], [5, 5]]) == 36

# Bài 3
def area_todo_ans(dimensions):
    best_sq = -1
    best_area = 0
    for l, w in dimensions:
        d_sq = l * l + w * w
        area = l * w
        if d_sq > best_sq:
            best_sq, best_area = d_sq, area
        elif d_sq == best_sq and area > best_area:
            best_area = area
    return best_area

for dims, expected in examples:
    assert area_todo_ans(dims) == expected

# Bài 4
def area_two_pass2(dimensions):
    m = max(l * l + w * w for l, w in dimensions)
    return max(l * w for l, w in dimensions if l * l + w * w == m)

for dims, expected in examples:
    assert area_two_pass2(dims) == expected

# Bài 5(a)
def top_k_rectangles(dimensions, k):
    return sorted(dimensions, key=lambda p: (p[0] ** 2 + p[1] ** 2, p[0] * p[1]), reverse=True)[:k]

assert top_k_rectangles([[9, 3], [8, 6], [1, 1]], 2) == [[8, 6], [9, 3]]
assert top_k_rectangles([[1, 7], [5, 5]], 1) == [[5, 5]]

# Bài 5(b)
def count_ties(dimensions):
    m = max(l * l + w * w for l, w in dimensions)
    return sum(1 for l, w in dimensions if l * l + w * w == m)

assert count_ties([[3, 4], [4, 3], [1, 1]]) == 2
assert count_ties([[9, 3], [8, 6]]) == 1

# Bài 5(c)
def max_diagonal_with_area_limit(dimensions, T):
    ok = [(l * l + w * w, l * w) for l, w in dimensions if l * w <= T]
    return max(ok)[1] if ok else 0

assert max_diagonal_with_area_limit([[9, 3], [8, 6]], 30) == 27       # 8x6 (48) bị loại, còn 9x3
assert max_diagonal_with_area_limit([[9, 3], [8, 6]], 10) == 0
assert max_diagonal_with_area_limit([[9, 3], [8, 6]], 100) == 48
print("PASS đáp án phần 19")

PASS đáp án phần 19


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (so sánh `(d², area)` theo từ điển bằng số nguyên)

```python
from typing import List

class Solution:
    def areaOfMaxDiagonal(self, dimensions: List[List[int]]) -> int:
        return max((l * l + w * w, l * w) for l, w in dimensions)[1]
```

**Checklist**

- [ ] So sánh **đường chéo** (qua `l² + w²`), không phải `l + w` hay diện tích?
- [ ] Dùng **số nguyên** (bình phương) thay vì `sqrt` và `==` trên số thực?
- [ ] Khi **hòa đường chéo**, giữ **diện tích lớn hơn** (không ghi đè bằng hình xuất hiện sau)?
- [ ] Khởi tạo `best` nhỏ hơn mọi khả năng (`-1`)?
- [ ] Trả về **diện tích** (số nguyên), không phải đường chéo?
- [ ] Đã thử: một hình, hòa hoàn toàn, hòa đường chéo khác diện tích (đổi thứ tự), hình dài mỏng `1 x 100`?
- [ ] Skeleton đúng `class Solution` / `def areaOfMaxDiagonal(self, dimensions: List[List[int]]) -> int`?